# Metrics Audit

Aggregate metrics files across experiment runs.

Steps:
- Scan for metrics.json and metrics.csv files.
- Summarize metrics keys.
- Highlight missing or empty metrics.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

summary = {
    'metrics_files': [],
    'missing': [],
}

experiments_root = REPO_ROOT / 'experiments'
if not experiments_root.exists():
    print('Missing:', experiments_root)
else:
    metrics_files = [p for p in experiments_root.rglob('*') if p.is_file() and p.name.startswith('metrics')]
    summary['metrics_files'] = [str(p.relative_to(REPO_ROOT)) for p in metrics_files]
    print('Metrics files:', len(metrics_files))

    for path in metrics_files:
        if path.suffix == '.json':
            try:
                data = json.loads(path.read_text(encoding='utf-8'))
                if not data:
                    summary['missing'].append(str(path.relative_to(REPO_ROOT)))
                else:
                    print(path.relative_to(REPO_ROOT), list(data.keys())[:6])
            except Exception as exc:
                summary['missing'].append(str(path.relative_to(REPO_ROOT)))
                print('Failed to parse', path, exc)
        elif path.suffix == '.csv':
            try:
                import pandas as pd
                df = pd.read_csv(path)
                if df.empty:
                    summary['missing'].append(str(path.relative_to(REPO_ROOT)))
                else:
                    print(path.relative_to(REPO_ROOT), 'rows:', df.shape[0])
            except Exception as exc:
                summary['missing'].append(str(path.relative_to(REPO_ROOT)))
                print('Failed to parse', path, exc)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_metrics_audit_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
